<a href="https://colab.research.google.com/github/Nidhi-578/PySpark_Assignments/blob/main/PySpark_Day2_Transformations_Retail_Banking_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark Day 2 — DataFrame Transformations
**Business dataset:** Retail orders  
**Assignment domain:** Banking

Goal: practice narrow/wide transformations and build business-ready retail outputs.

## Setup

In [57]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.appName("RetailPySparkTraining").getOrCreate()

retail_data = [
    (1001,"Laptop","Electronics",65000.0,2,0.10,"Bengaluru","2026-09-01"),
    (1002,"Phone","Electronics",30000.0,3,0.05,"Hyderabad","2026-09-02"),
    (1003,"Shoes","Fashion",4500.0,5,0.20,"Chennai","2026-09-02"),
    (1004,"Watch","Fashion",8000.0,2,0.15,"Bengaluru","2026-09-03"),
    (1005,"TV","Electronics",55000.0,1,0.10,"Pune","2026-09-03"),
    (1006,"Chair","Furniture",7000.0,4,0.05,"Hyderabad","2026-09-04"),
    (1007,"Table","Furniture",12000.0,2,0.10,"Chennai","2026-09-04"),
    (1008,"Headphones","Electronics",3500.0,6,0.00,"Bengaluru","2026-09-05"),
    (1009,"Bag","Fashion",2500.0,8,0.10,"Pune","2026-09-05"),
    (1010,"Sofa","Furniture",40000.0,1,0.12,"Bengaluru","2026-09-06")
]
cols=["order_id","product","category","price","quantity","discount","city","order_date"]
df=spark.createDataFrame(retail_data,cols)
df.show(truncate=False)

+--------+----------+-----------+-------+--------+--------+---------+----------+
|order_id|product   |category   |price  |quantity|discount|city     |order_date|
+--------+----------+-----------+-------+--------+--------+---------+----------+
|1001    |Laptop    |Electronics|65000.0|2       |0.1     |Bengaluru|2026-09-01|
|1002    |Phone     |Electronics|30000.0|3       |0.05    |Hyderabad|2026-09-02|
|1003    |Shoes     |Fashion    |4500.0 |5       |0.2     |Chennai  |2026-09-02|
|1004    |Watch     |Fashion    |8000.0 |2       |0.15    |Bengaluru|2026-09-03|
|1005    |TV        |Electronics|55000.0|1       |0.1     |Pune     |2026-09-03|
|1006    |Chair     |Furniture  |7000.0 |4       |0.05    |Hyderabad|2026-09-04|
|1007    |Table     |Furniture  |12000.0|2       |0.1     |Chennai  |2026-09-04|
|1008    |Headphones|Electronics|3500.0 |6       |0.0     |Bengaluru|2026-09-05|
|1009    |Bag       |Fashion    |2500.0 |8       |0.1     |Pune     |2026-09-05|
|1010    |Sofa      |Furnitu

## 1. `select()` — choose required columns

In [58]:
df.select("order_id","product","price").show()

+--------+----------+-------+
|order_id|   product|  price|
+--------+----------+-------+
|    1001|    Laptop|65000.0|
|    1002|     Phone|30000.0|
|    1003|     Shoes| 4500.0|
|    1004|     Watch| 8000.0|
|    1005|        TV|55000.0|
|    1006|     Chair| 7000.0|
|    1007|     Table|12000.0|
|    1008|Headphones| 3500.0|
|    1009|       Bag| 2500.0|
|    1010|      Sofa|40000.0|
+--------+----------+-------+



## 2. `filter()` / `where()` — filter business records

In [59]:
df.filter(F.col("price") > 10000).show()
df.where(F.col("category") == "Electronics").show()

+--------+-------+-----------+-------+--------+--------+---------+----------+
|order_id|product|   category|  price|quantity|discount|     city|order_date|
+--------+-------+-----------+-------+--------+--------+---------+----------+
|    1001| Laptop|Electronics|65000.0|       2|     0.1|Bengaluru|2026-09-01|
|    1002|  Phone|Electronics|30000.0|       3|    0.05|Hyderabad|2026-09-02|
|    1005|     TV|Electronics|55000.0|       1|     0.1|     Pune|2026-09-03|
|    1007|  Table|  Furniture|12000.0|       2|     0.1|  Chennai|2026-09-04|
|    1010|   Sofa|  Furniture|40000.0|       1|    0.12|Bengaluru|2026-09-06|
+--------+-------+-----------+-------+--------+--------+---------+----------+

+--------+----------+-----------+-------+--------+--------+---------+----------+
|order_id|   product|   category|  price|quantity|discount|     city|order_date|
+--------+----------+-----------+-------+--------+--------+---------+----------+
|    1001|    Laptop|Electronics|65000.0|       2|    

## 3. `withColumn()` — derive gross and net sales

In [60]:
sales_df=(df
 .withColumn("gross_sales", F.col("price")*F.col("quantity"))
 .withColumn("net_sales", F.col("price")*F.col("quantity")*(1-F.col("discount"))))
sales_df.show()

+--------+----------+-----------+-------+--------+--------+---------+----------+-----------+---------+
|order_id|   product|   category|  price|quantity|discount|     city|order_date|gross_sales|net_sales|
+--------+----------+-----------+-------+--------+--------+---------+----------+-----------+---------+
|    1001|    Laptop|Electronics|65000.0|       2|     0.1|Bengaluru|2026-09-01|   130000.0| 117000.0|
|    1002|     Phone|Electronics|30000.0|       3|    0.05|Hyderabad|2026-09-02|    90000.0|  85500.0|
|    1003|     Shoes|    Fashion| 4500.0|       5|     0.2|  Chennai|2026-09-02|    22500.0|  18000.0|
|    1004|     Watch|    Fashion| 8000.0|       2|    0.15|Bengaluru|2026-09-03|    16000.0|  13600.0|
|    1005|        TV|Electronics|55000.0|       1|     0.1|     Pune|2026-09-03|    55000.0|  49500.0|
|    1006|     Chair|  Furniture| 7000.0|       4|    0.05|Hyderabad|2026-09-04|    28000.0|  26600.0|
|    1007|     Table|  Furniture|12000.0|       2|     0.1|  Chennai|2026

## 4. `withColumnRenamed()`

In [61]:
df.withColumnRenamed("city","customer_city").show(3)

+--------+-------+-----------+-------+--------+--------+-------------+----------+
|order_id|product|   category|  price|quantity|discount|customer_city|order_date|
+--------+-------+-----------+-------+--------+--------+-------------+----------+
|    1001| Laptop|Electronics|65000.0|       2|     0.1|    Bengaluru|2026-09-01|
|    1002|  Phone|Electronics|30000.0|       3|    0.05|    Hyderabad|2026-09-02|
|    1003|  Shoes|    Fashion| 4500.0|       5|     0.2|      Chennai|2026-09-02|
+--------+-------+-----------+-------+--------+--------+-------------+----------+
only showing top 3 rows


## 5. `drop()`

In [62]:
df.drop("discount").show(3)

+--------+-------+-----------+-------+--------+---------+----------+
|order_id|product|   category|  price|quantity|     city|order_date|
+--------+-------+-----------+-------+--------+---------+----------+
|    1001| Laptop|Electronics|65000.0|       2|Bengaluru|2026-09-01|
|    1002|  Phone|Electronics|30000.0|       3|Hyderabad|2026-09-02|
|    1003|  Shoes|    Fashion| 4500.0|       5|  Chennai|2026-09-02|
+--------+-------+-----------+-------+--------+---------+----------+
only showing top 3 rows


## 6. `distinct()` and `dropDuplicates()`

In [63]:
df.select("category").distinct().show()
df.dropDuplicates(["order_id"]).show()

+-----------+
|   category|
+-----------+
|    Fashion|
|Electronics|
|  Furniture|
+-----------+

+--------+----------+-----------+-------+--------+--------+---------+----------+
|order_id|   product|   category|  price|quantity|discount|     city|order_date|
+--------+----------+-----------+-------+--------+--------+---------+----------+
|    1001|    Laptop|Electronics|65000.0|       2|     0.1|Bengaluru|2026-09-01|
|    1002|     Phone|Electronics|30000.0|       3|    0.05|Hyderabad|2026-09-02|
|    1003|     Shoes|    Fashion| 4500.0|       5|     0.2|  Chennai|2026-09-02|
|    1004|     Watch|    Fashion| 8000.0|       2|    0.15|Bengaluru|2026-09-03|
|    1005|        TV|Electronics|55000.0|       1|     0.1|     Pune|2026-09-03|
|    1006|     Chair|  Furniture| 7000.0|       4|    0.05|Hyderabad|2026-09-04|
|    1007|     Table|  Furniture|12000.0|       2|     0.1|  Chennai|2026-09-04|
|    1008|Headphones|Electronics| 3500.0|       6|     0.0|Bengaluru|2026-09-05|
|    1009|

## 7. `orderBy()` / `sort()`

In [64]:
df.orderBy(F.col("price").desc()).show()

+--------+----------+-----------+-------+--------+--------+---------+----------+
|order_id|   product|   category|  price|quantity|discount|     city|order_date|
+--------+----------+-----------+-------+--------+--------+---------+----------+
|    1001|    Laptop|Electronics|65000.0|       2|     0.1|Bengaluru|2026-09-01|
|    1005|        TV|Electronics|55000.0|       1|     0.1|     Pune|2026-09-03|
|    1010|      Sofa|  Furniture|40000.0|       1|    0.12|Bengaluru|2026-09-06|
|    1002|     Phone|Electronics|30000.0|       3|    0.05|Hyderabad|2026-09-02|
|    1007|     Table|  Furniture|12000.0|       2|     0.1|  Chennai|2026-09-04|
|    1004|     Watch|    Fashion| 8000.0|       2|    0.15|Bengaluru|2026-09-03|
|    1006|     Chair|  Furniture| 7000.0|       4|    0.05|Hyderabad|2026-09-04|
|    1003|     Shoes|    Fashion| 4500.0|       5|     0.2|  Chennai|2026-09-02|
|    1008|Headphones|Electronics| 3500.0|       6|     0.0|Bengaluru|2026-09-05|
|    1009|       Bag|    Fas

## 8. `groupBy()` + aggregations

In [65]:
df.groupBy("category").agg(
    F.count("order_id").alias("orders"),
    F.sum("quantity").alias("units"),
    F.avg("price").alias("avg_price"),
    F.max("price").alias("max_price")
).show()

+-----------+------+-----+------------------+---------+
|   category|orders|units|         avg_price|max_price|
+-----------+------+-----+------------------+---------+
|    Fashion|     3|   15|            5000.0|   8000.0|
|Electronics|     4|   12|           38375.0|  65000.0|
|  Furniture|     3|    7|19666.666666666668|  40000.0|
+-----------+------+-----+------------------+---------+



## 9. Conditional transformation with `when/otherwise`

In [66]:
df.withColumn("price_band",
    F.when(F.col("price")>=50000,"Premium")
     .when(F.col("price")>=10000,"Mid")
     .otherwise("Budget")
).show()

+--------+----------+-----------+-------+--------+--------+---------+----------+----------+
|order_id|   product|   category|  price|quantity|discount|     city|order_date|price_band|
+--------+----------+-----------+-------+--------+--------+---------+----------+----------+
|    1001|    Laptop|Electronics|65000.0|       2|     0.1|Bengaluru|2026-09-01|   Premium|
|    1002|     Phone|Electronics|30000.0|       3|    0.05|Hyderabad|2026-09-02|       Mid|
|    1003|     Shoes|    Fashion| 4500.0|       5|     0.2|  Chennai|2026-09-02|    Budget|
|    1004|     Watch|    Fashion| 8000.0|       2|    0.15|Bengaluru|2026-09-03|    Budget|
|    1005|        TV|Electronics|55000.0|       1|     0.1|     Pune|2026-09-03|   Premium|
|    1006|     Chair|  Furniture| 7000.0|       4|    0.05|Hyderabad|2026-09-04|    Budget|
|    1007|     Table|  Furniture|12000.0|       2|     0.1|  Chennai|2026-09-04|       Mid|
|    1008|Headphones|Electronics| 3500.0|       6|     0.0|Bengaluru|2026-09-05|

## 10. String transformations

In [67]:
df.select(
    F.upper("product").alias("product_upper"),
    F.lower("category").alias("category_lower"),
    F.length("product").alias("name_length")
).show()

+-------------+--------------+-----------+
|product_upper|category_lower|name_length|
+-------------+--------------+-----------+
|       LAPTOP|   electronics|          6|
|        PHONE|   electronics|          5|
|        SHOES|       fashion|          5|
|        WATCH|       fashion|          5|
|           TV|   electronics|          2|
|        CHAIR|     furniture|          5|
|        TABLE|     furniture|          5|
|   HEADPHONES|   electronics|         10|
|          BAG|       fashion|          3|
|         SOFA|     furniture|          4|
+-------------+--------------+-----------+



## 11. Date transformations

In [68]:
date_df=df.withColumn("order_date",F.to_date("order_date"))
date_df.select("order_id","order_date",F.year("order_date").alias("year"),F.month("order_date").alias("month"),F.dayofmonth("order_date").alias("day")).show()

+--------+----------+----+-----+---+
|order_id|order_date|year|month|day|
+--------+----------+----+-----+---+
|    1001|2026-09-01|2026|    9|  1|
|    1002|2026-09-02|2026|    9|  2|
|    1003|2026-09-02|2026|    9|  2|
|    1004|2026-09-03|2026|    9|  3|
|    1005|2026-09-03|2026|    9|  3|
|    1006|2026-09-04|2026|    9|  4|
|    1007|2026-09-04|2026|    9|  4|
|    1008|2026-09-05|2026|    9|  5|
|    1009|2026-09-05|2026|    9|  5|
|    1010|2026-09-06|2026|    9|  6|
+--------+----------+----+-----+---+



## 12. Null handling transformations

In [69]:
df.fillna({"discount":0.0}).dropna(subset=["order_id","product"]).show()

+--------+----------+-----------+-------+--------+--------+---------+----------+
|order_id|   product|   category|  price|quantity|discount|     city|order_date|
+--------+----------+-----------+-------+--------+--------+---------+----------+
|    1001|    Laptop|Electronics|65000.0|       2|     0.1|Bengaluru|2026-09-01|
|    1002|     Phone|Electronics|30000.0|       3|    0.05|Hyderabad|2026-09-02|
|    1003|     Shoes|    Fashion| 4500.0|       5|     0.2|  Chennai|2026-09-02|
|    1004|     Watch|    Fashion| 8000.0|       2|    0.15|Bengaluru|2026-09-03|
|    1005|        TV|Electronics|55000.0|       1|     0.1|     Pune|2026-09-03|
|    1006|     Chair|  Furniture| 7000.0|       4|    0.05|Hyderabad|2026-09-04|
|    1007|     Table|  Furniture|12000.0|       2|     0.1|  Chennai|2026-09-04|
|    1008|Headphones|Electronics| 3500.0|       6|     0.0|Bengaluru|2026-09-05|
|    1009|       Bag|    Fashion| 2500.0|       8|     0.1|     Pune|2026-09-05|
|    1010|      Sofa|  Furni

## 13. `unionByName()`

In [70]:
extra=spark.createDataFrame([(1011,"Keyboard","Electronics",2500.0,3,0.05,"Chennai","2026-09-07")],df.columns)
df.unionByName(extra).show()

+--------+----------+-----------+-------+--------+--------+---------+----------+
|order_id|   product|   category|  price|quantity|discount|     city|order_date|
+--------+----------+-----------+-------+--------+--------+---------+----------+
|    1001|    Laptop|Electronics|65000.0|       2|     0.1|Bengaluru|2026-09-01|
|    1002|     Phone|Electronics|30000.0|       3|    0.05|Hyderabad|2026-09-02|
|    1003|     Shoes|    Fashion| 4500.0|       5|     0.2|  Chennai|2026-09-02|
|    1004|     Watch|    Fashion| 8000.0|       2|    0.15|Bengaluru|2026-09-03|
|    1005|        TV|Electronics|55000.0|       1|     0.1|     Pune|2026-09-03|
|    1006|     Chair|  Furniture| 7000.0|       4|    0.05|Hyderabad|2026-09-04|
|    1007|     Table|  Furniture|12000.0|       2|     0.1|  Chennai|2026-09-04|
|    1008|Headphones|Electronics| 3500.0|       6|     0.0|Bengaluru|2026-09-05|
|    1009|       Bag|    Fashion| 2500.0|       8|     0.1|     Pune|2026-09-05|
|    1010|      Sofa|  Furni

## 14. Join transformation

In [71]:
category_data=[("Electronics",0.18),("Fashion",0.12),("Furniture",0.15)]
category_df=spark.createDataFrame(category_data,["category","tax_rate"])
df.join(category_df,"category","left").show()

+-----------+--------+----------+-------+--------+--------+---------+----------+--------+
|   category|order_id|   product|  price|quantity|discount|     city|order_date|tax_rate|
+-----------+--------+----------+-------+--------+--------+---------+----------+--------+
|    Fashion|    1003|     Shoes| 4500.0|       5|     0.2|  Chennai|2026-09-02|    0.12|
|    Fashion|    1004|     Watch| 8000.0|       2|    0.15|Bengaluru|2026-09-03|    0.12|
|Electronics|    1001|    Laptop|65000.0|       2|     0.1|Bengaluru|2026-09-01|    0.18|
|Electronics|    1002|     Phone|30000.0|       3|    0.05|Hyderabad|2026-09-02|    0.18|
|Electronics|    1005|        TV|55000.0|       1|     0.1|     Pune|2026-09-03|    0.18|
|    Fashion|    1009|       Bag| 2500.0|       8|     0.1|     Pune|2026-09-05|    0.12|
|Electronics|    1008|Headphones| 3500.0|       6|     0.0|Bengaluru|2026-09-05|    0.18|
|  Furniture|    1006|     Chair| 7000.0|       4|    0.05|Hyderabad|2026-09-04|    0.15|
|  Furnitu

## 15. Repartition and coalesce

In [72]:
print("Original partitions:",df.rdd.getNumPartitions())
repart_df=df.repartition(4,"category")
print("After repartition:",repart_df.rdd.getNumPartitions())
coal_df=repart_df.coalesce(2)
print("After coalesce:",coal_df.rdd.getNumPartitions())

Original partitions: 2
After repartition: 4
After coalesce: 2


## 16. Retail transformation pipeline

In [73]:
final_df=(df
 .filter(F.col("quantity")>0)
 .withColumn("gross_sales",F.col("price")*F.col("quantity"))
 .withColumn("net_sales",F.round(F.col("gross_sales")*(1-F.col("discount")),2))
 .withColumn("order_date",F.to_date("order_date"))
 .select("order_id","product","category","city","order_date","quantity","gross_sales","net_sales")
 .orderBy(F.col("net_sales").desc()))
final_df.show(truncate=False)

+--------+----------+-----------+---------+----------+--------+-----------+---------+
|order_id|product   |category   |city     |order_date|quantity|gross_sales|net_sales|
+--------+----------+-----------+---------+----------+--------+-----------+---------+
|1001    |Laptop    |Electronics|Bengaluru|2026-09-01|2       |130000.0   |117000.0 |
|1002    |Phone     |Electronics|Hyderabad|2026-09-02|3       |90000.0    |85500.0  |
|1005    |TV        |Electronics|Pune     |2026-09-03|1       |55000.0    |49500.0  |
|1010    |Sofa      |Furniture  |Bengaluru|2026-09-06|1       |40000.0    |35200.0  |
|1006    |Chair     |Furniture  |Hyderabad|2026-09-04|4       |28000.0    |26600.0  |
|1007    |Table     |Furniture  |Chennai  |2026-09-04|2       |24000.0    |21600.0  |
|1008    |Headphones|Electronics|Bengaluru|2026-09-05|6       |21000.0    |21000.0  |
|1003    |Shoes     |Fashion    |Chennai  |2026-09-02|5       |22500.0    |18000.0  |
|1009    |Bag       |Fashion    |Pune     |2026-09-05|

## Day 2 Banking Assignment
Create/use a banking transactions DataFrame with: `transaction_id, customer_id, account_type, transaction_type, amount, city, transaction_date`.

Tasks:
1. Select transaction ID, customer ID and amount.
2. Filter transactions above ₹50,000.
3. Filter only `DEBIT` transactions.
4. Create `transaction_fee` as 1% of amount.
5. Create `net_amount` after fee.
6. Rename `city` to `branch_city`.
7. Drop an unnecessary column.
8. Find distinct account types.
9. Remove duplicate transaction IDs.
10. Sort transactions by amount descending.
11. Group by account type and calculate count, total amount, average amount, min and max.
12. Create `risk_level`: HIGH >= ₹100,000, MEDIUM >= ₹50,000, otherwise LOW.
13. Convert transaction date to DateType and extract year/month/day.
14. Handle null amount/city values appropriately.
15. Create another DataFrame and perform `unionByName`.
16. Create a customer master DataFrame and join it with transactions.
17. Repartition by account type and explain why.
18. Build a final banking pipeline producing clean, enriched transactions.

**Challenge:** Classify the transformations you used as narrow or wide and identify where a shuffle is likely to occur.

In [74]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("BankingPySparkTraining") \
    .getOrCreate()

banking_data = [
    (2001, 101, "SAVINGS", "CREDIT", 75000.0, "Bengaluru", "2026-09-01"),
    (2002, 102, "CURRENT", "DEBIT", 120000.0, "Hyderabad", "2026-09-02"),
    (2003, 103, "SAVINGS", "DEBIT", 45000.0, "Chennai", "2026-09-02"),
    (2004, 104, "SALARY", "CREDIT", 85000.0, "Bengaluru", "2026-09-03"),
    (2005, 105, "SAVINGS", "DEBIT", 150000.0, "Pune", "2026-09-03"),
    (2006, 106, "CURRENT", "CREDIT", 25000.0, "Hyderabad", "2026-09-04"),
    (2007, 107, "SALARY", "DEBIT", 60000.0, "Chennai", "2026-09-04"),
    (2008, 108, "SAVINGS", "CREDIT", 35000.0, "Bengaluru", "2026-09-05"),
    (2009, 109, "CURRENT", "DEBIT", 95000.0, "Pune", "2026-09-05"),
    (2010, 110, "SAVINGS", "DEBIT", 200000.0, "Bengaluru", "2026-09-06")
]

cols = [
    "transaction_id",
    "customer_id",
    "account_type",
    "transaction_type",
    "amount",
    "city",
    "transaction_date"
]

banking_df = spark.createDataFrame(
    banking_data,
    cols
)

banking_df.show(truncate=False)

+--------------+-----------+------------+----------------+--------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|amount  |city     |transaction_date|
+--------------+-----------+------------+----------------+--------+---------+----------------+
|2001          |101        |SAVINGS     |CREDIT          |75000.0 |Bengaluru|2026-09-01      |
|2002          |102        |CURRENT     |DEBIT           |120000.0|Hyderabad|2026-09-02      |
|2003          |103        |SAVINGS     |DEBIT           |45000.0 |Chennai  |2026-09-02      |
|2004          |104        |SALARY      |CREDIT          |85000.0 |Bengaluru|2026-09-03      |
|2005          |105        |SAVINGS     |DEBIT           |150000.0|Pune     |2026-09-03      |
|2006          |106        |CURRENT     |CREDIT          |25000.0 |Hyderabad|2026-09-04      |
|2007          |107        |SALARY      |DEBIT           |60000.0 |Chennai  |2026-09-04      |
|2008          |108        |SAVINGS     |CREDIT   

Select transaction ID, customer ID and amount.

In [75]:
banking_df.select("transaction_id","customer_id","amount").show()

+--------------+-----------+--------+
|transaction_id|customer_id|  amount|
+--------------+-----------+--------+
|          2001|        101| 75000.0|
|          2002|        102|120000.0|
|          2003|        103| 45000.0|
|          2004|        104| 85000.0|
|          2005|        105|150000.0|
|          2006|        106| 25000.0|
|          2007|        107| 60000.0|
|          2008|        108| 35000.0|
|          2009|        109| 95000.0|
|          2010|        110|200000.0|
+--------------+-----------+--------+



Filter transactions above ₹50,000.

In [76]:
banking_df.filter(F.col("amount")>50000).show()

+--------------+-----------+------------+----------------+--------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|
+--------------+-----------+------------+----------------+--------+---------+----------------+
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|
|          2002|        102|     CURRENT|           DEBIT|120000.0|Hyderabad|      2026-09-02|
|          2004|        104|      SALARY|          CREDIT| 85000.0|Bengaluru|      2026-09-03|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|     Pune|      2026-09-03|
|          2007|        107|      SALARY|           DEBIT| 60000.0|  Chennai|      2026-09-04|
|          2009|        109|     CURRENT|           DEBIT| 95000.0|     Pune|      2026-09-05|
|          2010|        110|     SAVINGS|           DEBIT|200000.0|Bengaluru|      2026-09-06|
+--------------+-----------+------------+---------

3. Filter only DEBIT transactions.

In [77]:
banking_df.filter(F.col("transaction_type") == "DEBIT").show()

+--------------+-----------+------------+----------------+--------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|
+--------------+-----------+------------+----------------+--------+---------+----------------+
|          2002|        102|     CURRENT|           DEBIT|120000.0|Hyderabad|      2026-09-02|
|          2003|        103|     SAVINGS|           DEBIT| 45000.0|  Chennai|      2026-09-02|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|     Pune|      2026-09-03|
|          2007|        107|      SALARY|           DEBIT| 60000.0|  Chennai|      2026-09-04|
|          2009|        109|     CURRENT|           DEBIT| 95000.0|     Pune|      2026-09-05|
|          2010|        110|     SAVINGS|           DEBIT|200000.0|Bengaluru|      2026-09-06|
+--------------+-----------+------------+----------------+--------+---------+----------------+



4. Create transaction_fee as 1% of amount.

In [78]:
banking_df.withColumn("transaction_fee",F.col("amount")*0.01).show()

+--------------+-----------+------------+----------------+--------+---------+----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|transaction_fee|
+--------------+-----------+------------+----------------+--------+---------+----------------+---------------+
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|          750.0|
|          2002|        102|     CURRENT|           DEBIT|120000.0|Hyderabad|      2026-09-02|         1200.0|
|          2003|        103|     SAVINGS|           DEBIT| 45000.0|  Chennai|      2026-09-02|          450.0|
|          2004|        104|      SALARY|          CREDIT| 85000.0|Bengaluru|      2026-09-03|          850.0|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|     Pune|      2026-09-03|         1500.0|
|          2006|        106|     CURRENT|          CREDIT| 25000.0|Hyderabad|      2026-09-04|          250.0|
|

5. Create net_amount after fee.

In [79]:
banking_df = banking_df.withColumn(
    "transaction_fee",
    F.round(F.col("amount") * 0.01, 2)
)
banking_df.show()

+--------------+-----------+------------+----------------+--------+---------+----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|transaction_fee|
+--------------+-----------+------------+----------------+--------+---------+----------------+---------------+
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|          750.0|
|          2002|        102|     CURRENT|           DEBIT|120000.0|Hyderabad|      2026-09-02|         1200.0|
|          2003|        103|     SAVINGS|           DEBIT| 45000.0|  Chennai|      2026-09-02|          450.0|
|          2004|        104|      SALARY|          CREDIT| 85000.0|Bengaluru|      2026-09-03|          850.0|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|     Pune|      2026-09-03|         1500.0|
|          2006|        106|     CURRENT|          CREDIT| 25000.0|Hyderabad|      2026-09-04|          250.0|
|

6. Rename city to branch_city.

In [80]:
banking_df = banking_df.withColumnRenamed(
    "city",
    "branch_city"
)
banking_df.show()

+--------------+-----------+------------+----------------+--------+-----------+----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|branch_city|transaction_date|transaction_fee|
+--------------+-----------+------------+----------------+--------+-----------+----------------+---------------+
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|  Bengaluru|      2026-09-01|          750.0|
|          2002|        102|     CURRENT|           DEBIT|120000.0|  Hyderabad|      2026-09-02|         1200.0|
|          2003|        103|     SAVINGS|           DEBIT| 45000.0|    Chennai|      2026-09-02|          450.0|
|          2004|        104|      SALARY|          CREDIT| 85000.0|  Bengaluru|      2026-09-03|          850.0|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|       Pune|      2026-09-03|         1500.0|
|          2006|        106|     CURRENT|          CREDIT| 25000.0|  Hyderabad|      2026-09-04|

7. Drop an unnecessary column.

In [81]:
banking_df = banking_df.drop("transaction_fee")
banking_df.show()

+--------------+-----------+------------+----------------+--------+-----------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|branch_city|transaction_date|
+--------------+-----------+------------+----------------+--------+-----------+----------------+
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|  Bengaluru|      2026-09-01|
|          2002|        102|     CURRENT|           DEBIT|120000.0|  Hyderabad|      2026-09-02|
|          2003|        103|     SAVINGS|           DEBIT| 45000.0|    Chennai|      2026-09-02|
|          2004|        104|      SALARY|          CREDIT| 85000.0|  Bengaluru|      2026-09-03|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|       Pune|      2026-09-03|
|          2006|        106|     CURRENT|          CREDIT| 25000.0|  Hyderabad|      2026-09-04|
|          2007|        107|      SALARY|           DEBIT| 60000.0|    Chennai|      2026-09-04|
|          2008|        108|  

8. Find distinct account types.

In [82]:
banking_df.select("account_type").distinct().show()

+------------+
|account_type|
+------------+
|      SALARY|
|     SAVINGS|
|     CURRENT|
+------------+



9. Remove duplicate transaction IDs.

In [83]:
banking_df = banking_df.dropDuplicates(["transaction_id"])
banking_df.show()

+--------------+-----------+------------+----------------+--------+-----------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|branch_city|transaction_date|
+--------------+-----------+------------+----------------+--------+-----------+----------------+
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|  Bengaluru|      2026-09-01|
|          2002|        102|     CURRENT|           DEBIT|120000.0|  Hyderabad|      2026-09-02|
|          2003|        103|     SAVINGS|           DEBIT| 45000.0|    Chennai|      2026-09-02|
|          2004|        104|      SALARY|          CREDIT| 85000.0|  Bengaluru|      2026-09-03|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|       Pune|      2026-09-03|
|          2006|        106|     CURRENT|          CREDIT| 25000.0|  Hyderabad|      2026-09-04|
|          2007|        107|      SALARY|           DEBIT| 60000.0|    Chennai|      2026-09-04|
|          2008|        108|  

10. Sort transactions by amount descending.

In [84]:
banking_df.orderBy(F.col("amount").desc()).show()

+--------------+-----------+------------+----------------+--------+-----------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|branch_city|transaction_date|
+--------------+-----------+------------+----------------+--------+-----------+----------------+
|          2010|        110|     SAVINGS|           DEBIT|200000.0|  Bengaluru|      2026-09-06|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|       Pune|      2026-09-03|
|          2002|        102|     CURRENT|           DEBIT|120000.0|  Hyderabad|      2026-09-02|
|          2009|        109|     CURRENT|           DEBIT| 95000.0|       Pune|      2026-09-05|
|          2004|        104|      SALARY|          CREDIT| 85000.0|  Bengaluru|      2026-09-03|
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|  Bengaluru|      2026-09-01|
|          2007|        107|      SALARY|           DEBIT| 60000.0|    Chennai|      2026-09-04|
|          2003|        103|  

11. Group by account type and calculate count, total amount, average amount, min and max.

In [85]:
account_summary = banking_df.groupBy("account_type").agg(
    F.count("transaction_id").alias("transaction_count"),
    F.sum("amount").alias("total_amount"),
    F.avg("amount").alias("average_amount"),
    F.min("amount").alias("minimum_amount"),
    F.max("amount").alias("maximum_amount")
)

account_summary.show()

+------------+-----------------+------------+--------------+--------------+--------------+
|account_type|transaction_count|total_amount|average_amount|minimum_amount|maximum_amount|
+------------+-----------------+------------+--------------+--------------+--------------+
|      SALARY|                2|    145000.0|       72500.0|       60000.0|       85000.0|
|     SAVINGS|                5|    505000.0|      101000.0|       35000.0|      200000.0|
|     CURRENT|                3|    240000.0|       80000.0|       25000.0|      120000.0|
+------------+-----------------+------------+--------------+--------------+--------------+



12. Create risk_level: HIGH >= ₹100,000, MEDIUM >= ₹50,000, otherwise LOW.

In [86]:
banking_df = banking_df.withColumn(
    "risk_level",
    F.when(
        F.col("amount") >= 100000,
        "HIGH"
    ).when(
        F.col("amount") >= 50000,
        "MEDIUM"
    ).otherwise(
        "LOW"
    )
)

banking_df.show()

+--------------+-----------+------------+----------------+--------+-----------+----------------+----------+
|transaction_id|customer_id|account_type|transaction_type|  amount|branch_city|transaction_date|risk_level|
+--------------+-----------+------------+----------------+--------+-----------+----------------+----------+
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|  Bengaluru|      2026-09-01|    MEDIUM|
|          2002|        102|     CURRENT|           DEBIT|120000.0|  Hyderabad|      2026-09-02|      HIGH|
|          2003|        103|     SAVINGS|           DEBIT| 45000.0|    Chennai|      2026-09-02|       LOW|
|          2004|        104|      SALARY|          CREDIT| 85000.0|  Bengaluru|      2026-09-03|    MEDIUM|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|       Pune|      2026-09-03|      HIGH|
|          2006|        106|     CURRENT|          CREDIT| 25000.0|  Hyderabad|      2026-09-04|       LOW|
|          2007|        107|

13. Convert transaction date to DateType and extract year/month/day.

In [87]:
banking_df.select(
    "transaction_id",
    "transaction_date",
    F.year("transaction_date").alias("year"),
    F.month("transaction_date").alias("month"),
    F.dayofmonth("transaction_date").alias("day")
).show()

+--------------+----------------+----+-----+---+
|transaction_id|transaction_date|year|month|day|
+--------------+----------------+----+-----+---+
|          2001|      2026-09-01|2026|    9|  1|
|          2002|      2026-09-02|2026|    9|  2|
|          2003|      2026-09-02|2026|    9|  2|
|          2004|      2026-09-03|2026|    9|  3|
|          2005|      2026-09-03|2026|    9|  3|
|          2006|      2026-09-04|2026|    9|  4|
|          2007|      2026-09-04|2026|    9|  4|
|          2008|      2026-09-05|2026|    9|  5|
|          2009|      2026-09-05|2026|    9|  5|
|          2010|      2026-09-06|2026|    9|  6|
+--------------+----------------+----+-----+---+



14. Handle null amount/city values appropriately.

In [88]:
banking_df = banking_df.fillna({
    "amount": 0.0,
    "branch_city": "Unknown"
})

banking_df.show()

+--------------+-----------+------------+----------------+--------+-----------+----------------+----------+
|transaction_id|customer_id|account_type|transaction_type|  amount|branch_city|transaction_date|risk_level|
+--------------+-----------+------------+----------------+--------+-----------+----------------+----------+
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|  Bengaluru|      2026-09-01|    MEDIUM|
|          2002|        102|     CURRENT|           DEBIT|120000.0|  Hyderabad|      2026-09-02|      HIGH|
|          2003|        103|     SAVINGS|           DEBIT| 45000.0|    Chennai|      2026-09-02|       LOW|
|          2004|        104|      SALARY|          CREDIT| 85000.0|  Bengaluru|      2026-09-03|    MEDIUM|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|       Pune|      2026-09-03|      HIGH|
|          2006|        106|     CURRENT|          CREDIT| 25000.0|  Hyderabad|      2026-09-04|       LOW|
|          2007|        107|

15. Create another DataFrame and perform unionByName.

In [89]:
extra_data = [
    (2011, 111, "SAVINGS", "CREDIT", 90000.0, "Mumbai", "2026-09-07")
]

extra_df = spark.createDataFrame(
    extra_data,
    [
        "transaction_id",
        "customer_id",
        "account_type",
        "transaction_type",
        "amount",
        "branch_city",
        "transaction_date"
    ]
)

extra_df.show()

+--------------+-----------+------------+----------------+-------+-----------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|branch_city|transaction_date|
+--------------+-----------+------------+----------------+-------+-----------+----------------+
|          2011|        111|     SAVINGS|          CREDIT|90000.0|     Mumbai|      2026-09-07|
+--------------+-----------+------------+----------------+-------+-----------+----------------+



In [90]:
banking_df = banking_df.unionByName(
    extra_df,
    allowMissingColumns=True
)
banking_df.show()

+--------------+-----------+------------+----------------+--------+-----------+----------------+----------+
|transaction_id|customer_id|account_type|transaction_type|  amount|branch_city|transaction_date|risk_level|
+--------------+-----------+------------+----------------+--------+-----------+----------------+----------+
|          2001|        101|     SAVINGS|          CREDIT| 75000.0|  Bengaluru|      2026-09-01|    MEDIUM|
|          2002|        102|     CURRENT|           DEBIT|120000.0|  Hyderabad|      2026-09-02|      HIGH|
|          2003|        103|     SAVINGS|           DEBIT| 45000.0|    Chennai|      2026-09-02|       LOW|
|          2004|        104|      SALARY|          CREDIT| 85000.0|  Bengaluru|      2026-09-03|    MEDIUM|
|          2005|        105|     SAVINGS|           DEBIT|150000.0|       Pune|      2026-09-03|      HIGH|
|          2006|        106|     CURRENT|          CREDIT| 25000.0|  Hyderabad|      2026-09-04|       LOW|
|          2007|        107|

Create a customer master DataFrame and join it with transactions.

In [91]:
customer_data = [
    (101, "Nidhi", "Premium"),
    (102, "Rahul", "Gold"),
    (103, "Amit", "Silver"),
    (104, "Priya", "Premium"),
    (105, "Neha", "Gold"),
    (106, "Ravi", "Silver"),
    (107, "Ankit", "Gold"),
    (108, "Sneha", "Premium"),
    (109, "Karan", "Silver"),
    (110, "Pooja", "Premium"),
    (111, "Riya", "Gold")
]

customer_cols = [
    "customer_id",
    "customer_name",
    "customer_segment"
]

customer_df = spark.createDataFrame(
    customer_data,
    customer_cols
)

customer_df.show()

+-----------+-------------+----------------+
|customer_id|customer_name|customer_segment|
+-----------+-------------+----------------+
|        101|        Nidhi|         Premium|
|        102|        Rahul|            Gold|
|        103|         Amit|          Silver|
|        104|        Priya|         Premium|
|        105|         Neha|            Gold|
|        106|         Ravi|          Silver|
|        107|        Ankit|            Gold|
|        108|        Sneha|         Premium|
|        109|        Karan|          Silver|
|        110|        Pooja|         Premium|
|        111|         Riya|            Gold|
+-----------+-------------+----------------+



In [92]:
joined_df = banking_df.join(
    customer_df,
    on="customer_id",
    how="left"
)

joined_df.show(truncate=False)

+-----------+--------------+------------+----------------+--------+-----------+----------------+----------+-------------+----------------+
|customer_id|transaction_id|account_type|transaction_type|amount  |branch_city|transaction_date|risk_level|customer_name|customer_segment|
+-----------+--------------+------------+----------------+--------+-----------+----------------+----------+-------------+----------------+
|110        |2010          |SAVINGS     |DEBIT           |200000.0|Bengaluru  |2026-09-06      |HIGH      |Pooja        |Premium         |
|107        |2007          |SALARY      |DEBIT           |60000.0 |Chennai    |2026-09-04      |MEDIUM    |Ankit        |Gold            |
|103        |2003          |SAVINGS     |DEBIT           |45000.0 |Chennai    |2026-09-02      |LOW       |Amit         |Silver          |
|104        |2004          |SALARY      |CREDIT          |85000.0 |Bengaluru  |2026-09-03      |MEDIUM    |Priya        |Premium         |
|106        |2006          

Repartition by account type and explain why.

In [93]:
repartitioned_df = banking_df.repartition(
    4,
    "account_type"
)

print(
    "Original partitions:",
    banking_df.rdd.getNumPartitions()
)

print(
    "After repartition:",
    repartitioned_df.rdd.getNumPartitions()
)

Original partitions: 3
After repartition: 4


Build a final banking pipeline producing clean, enriched transactions.

In [94]:
from pyspark.sql import functions as F

banking_data = [
    (2001, 101, "SAVINGS", "CREDIT", 75000.0, "Bengaluru", "2026-09-01"),
    (2002, 102, "CURRENT", "DEBIT", 120000.0, "Hyderabad", "2026-09-02"),
    (2003, 103, "SAVINGS", "DEBIT", 45000.0, "Chennai", "2026-09-02"),
    (2004, 104, "SALARY", "CREDIT", 85000.0, "Bengaluru", "2026-09-03"),
    (2005, 105, "SAVINGS", "DEBIT", 150000.0, "Pune", "2026-09-03"),
    (2006, 106, "CURRENT", "CREDIT", 25000.0, "Hyderabad", "2026-09-04"),
    (2007, 107, "SALARY", "DEBIT", 60000.0, "Chennai", "2026-09-04"),
    (2008, 108, "SAVINGS", "CREDIT", 35000.0, "Bengaluru", "2026-09-05"),
    (2009, 109, "CURRENT", "DEBIT", 95000.0, "Pune", "2026-09-05"),
    (2010, 110, "SAVINGS", "DEBIT", 200000.0, "Bengaluru", "2026-09-06")
]

cols = [
    "transaction_id",
    "customer_id",
    "account_type",
    "transaction_type",
    "amount",
    "city",
    "transaction_date"
]

banking_df = spark.createDataFrame(banking_data, cols)

customer_data = [
    (101, "Nidhi", "Premium"),
    (102, "Rahul", "Gold"),
    (103, "Amit", "Silver"),
    (104, "Priya", "Premium"),
    (105, "Neha", "Gold"),
    (106, "Ravi", "Silver"),
    (107, "Ankit", "Gold"),
    (108, "Sneha", "Premium"),
    (109, "Karan", "Silver"),
    (110, "Pooja", "Premium")
]

customer_cols = [
    "customer_id",
    "customer_name",
    "customer_segment"
]

customer_df = spark.createDataFrame(
    customer_data,
    customer_cols
)


final_banking_df = (
    banking_df


    .withColumnRenamed("city", "branch_city")


    .filter(F.col("amount").isNotNull())
    .filter(F.col("amount") > 0)

    .dropDuplicates(["transaction_id"])
    .withColumn(
        "transaction_fee",
        F.round(F.col("amount") * 0.01, 2)
    )

    .withColumn(
        "net_amount",F.round(F.col("amount") - F.col("transaction_fee"),2)
    )

    .withColumn(
        "risk_level",
        F.when(F.col("amount") >= 100000, "HIGH")
         .when(F.col("amount") >= 50000, "MEDIUM")
         .otherwise("LOW")
    )

    .withColumn(
        "transaction_date",
        F.to_date("transaction_date")
    )


    .withColumn(
        "year",
        F.year("transaction_date")
    )
    .withColumn(
        "month",
        F.month("transaction_date")
    )
    .withColumn(
        "day",
        F.dayofmonth("transaction_date")
    )

    .fillna({
        "branch_city": "Unknown"
    })

    # Join with customer master
    .join(
        F.broadcast(customer_df),
        on="customer_id",
        how="left"
    )

    # Select final business columns
    .select(
        "transaction_id",
        "customer_id",
        "customer_name",
        "customer_segment",
        "account_type",
        "transaction_type",
        "amount",
        "transaction_fee",
        "net_amount",
        "branch_city",
        "transaction_date",
        "year",
        "month",
        "day",
        "risk_level"
    )
)


final_banking_df.show(truncate=False)

+--------------+-----------+-------------+----------------+------------+----------------+--------+---------------+----------+-----------+----------------+----+-----+---+----------+
|transaction_id|customer_id|customer_name|customer_segment|account_type|transaction_type|amount  |transaction_fee|net_amount|branch_city|transaction_date|year|month|day|risk_level|
+--------------+-----------+-------------+----------------+------------+----------------+--------+---------------+----------+-----------+----------------+----+-----+---+----------+
|2001          |101        |Nidhi        |Premium         |SAVINGS     |CREDIT          |75000.0 |750.0          |74250.0   |Bengaluru  |2026-09-01      |2026|9    |1  |MEDIUM    |
|2002          |102        |Rahul        |Gold            |CURRENT     |DEBIT           |120000.0|1200.0         |118800.0  |Hyderabad  |2026-09-02      |2026|9    |2  |HIGH      |
|2003          |103        |Amit         |Silver          |SAVINGS     |DEBIT           |45000.

Challenge: Classify the transformations you used as narrow or wide and identify where a shuffle is likely to occur.

Narrow transformations: select(), filter(), where(), withColumn(), withColumnRenamed(), drop(), when/otherwise, date functions, fillna(), and unionByName(). These generally do not require shuffle because each output partition can be processed independently.
Wide transformations: distinct(), dropDuplicates(), groupBy(), orderBy(), repartition(), and normal joins. These generally require shuffle because data has to be redistributed between partitions based on a key or ordering.
In my banking pipeline, shuffle is likely during duplicate removal, grouping, sorting, repartitioning, and a normal join. I used a broadcast join for the small customer master to potentially avoid shuffling the large transaction dataset.